# Access Control Frameowrk

SF is based on:
### DAC - Discretionary access control 

Each object has an owner, who can in turn grant access to that object.

### RBAC - Role-Based Access Control

Access privileges are assigned to roles, which are in turn assigned to users

### UBAC - User-based Access Control :
Only considered when executing: Use secondary role is set to all.

USE SECONDARY ROLES ALL;

Still grants the ownershipt to the primary role.


# Roles

## System-defined Roles:

- GLOBALORGADMIN: organization-level tasks - only in organization accounts.
    - ORGADMIN: Phase-out - use GLOBALORGADMIN instead
- ACCOUNTADMIN: encapsulates the SYSADMIN and SECURITYADMIN system-defined roles
- SECURITYADMIN: Can manage any object grant globally, as well as create, monitor, and manage users and roles
- USERADMIN: dedicated to user and role management only
- SYSADMIN: create warehouses and databases (and other objects) in an account
- PUBLIC: Pseudo-role that is automatically granted to every user and every role in your account

System-defined roles cannot be dropped. In addition, the privileges granted to these roles by Snowflake cannot be revoked.


![alt](https://docs.snowflake.com/en/_images/system-role-hierarchy.png)


## GLOBALORGADMIN

This role manages organization-wide configurations, including account-level changes like renaming and setting up URL redirection for continuity.

### Organization

An organization is a first-class Snowflake object that links the accounts owned by your business entity

### Types of accounts

- Organization account: Special account used by organization administrators to manage multi-account organizations and to access usage data from premium views in the ORGANIZATION_USAGE schema.

- Regular Snowflake account, including trial accounts.

- Snowflake Open Catalog account: Special account used by service admins and catalog admins to manage catalogs defined in Snowflake Open Catalog.

### Enabling the ORGADMIN role in an account

The first account in an organization has the ORGADMIN role enabled. You can use this account to enable the role in other accounts.

    USE ROLE ORGADMIN;
    ALTER ACCOUNT my_account1 SET IS_ORG_ADMIN = TRUE;

### Disable the ORGADMIN role

    ALTER ACCOUNT account_123 SET IS_ORG_ADMIN = FALSE;

## ACCOUNTADMIN

- the first user is assigned the ACCOUNTADMIN role.
- View and manage Snowflake billing and credit data, and can stop any running SQL statements
- Not a superuser role. This role only allows viewing and managing objects in the account if this role, or a role lower in a role hierarchy, has sufficient privileges on the objects
- This user should then create one or more additional users who are assigned the USERADMIN role. 
- All remaining users should be created by the user(s) with the USERADMIN role or another role that is granted the global CREATE USER privilege.

Children:
- USERADMIN
- SECURITYADMIN 
- SYSADMIN

### Snowflake recommendations

- Assign this role only to a select/limited number of people in your organization.
- All users assigned the ACCOUNTADMIN role should also be required to use multi-factor authentication (MFA) for login.
- Assign this role to at least two users. We follow strict security procedures for resetting a forgotten or lost password for users with the ACCOUNTADMIN role. These procedures can take up to two business days.


### Enforce MFA enrollment on a human ACCOUNTADMIN

If a human directly uses the ACCOUNTADMIN role on your account, you can secure your account by forcing this account administrator to enroll in MFA during account creation

    CREATE ACCOUNT my_admin ADMIN_USER_TYPE = PERSON;

### Prevent MFA from being enforced on a non-human

If a human does not use the ACCOUNTADMIN role on your account, you must prevent MFA enrollment

    CREATE ACCOUNT my_admin
      ADMIN_USER_TYPE = SERVICE
      ADMIN_RSA_PUBLIC_KEY = 'MIIBIj...';

In [ ]:
show accounts;

In [ ]:
%%sql -r dataframe_1
use role sysadmin;
create role test;

In [ ]:
%%sql -r dataframe_2
-- to create a role, min requirment is securityadmin

use role securityadmin;
create role test;

In [ ]:
%%sql -r dataframe_3
use role securityadmin;
drop role test;
SHOW ROLES;

In [ ]:
%%sql -r dataframe_4
-- Set up the marketing database
USE ROLE SYSADMIN;

CREATE DATABASE MARKETING;
CREATE SCHEMA SALES;
CREATE TABLE MARKETING.SALES.CAMPAIGN (ID INT, COST NUMERIC,SALES_AMOUNT NUMERIC);


In [ ]:
%%sql -r dataframe_5
-- Use USERADMIN or SECURITYADMIN to set up role
USE ROLE SECURITYADMIN;

-- Create role and user
CREATE ROLE MARKETING_ADMIN;

CREATE USER INITIAL_USER
  PASSWORD = 'AbC201§#';

-- Assign user  
GRANT ROLE MARKETING_ADMIN TO USER INITIAL_USER;

-- Assign role to SYSADMIN -> important to keep the role hierarchy
GRANT ROLE MARKETING_ADMIN TO ROLE SYSADMIN;


## Object Hierarchy

- to grant to objects, you have to also grant usage to the higher objects.
    - ie: granting select access to a table, you are required to grant usage in database and schema(higher objects)

![image info](https://docs.snowflake.com/en/_images/securable-objects-hierarchy.png)



## Role hierachy

Snowflake recommends creating a hierarchy of custom roles, with the top-most custom role assigned to the system role SYSADMIN.

***if a custom role is not assigned to SYSADMIN through a role hierarchy, the system administrators cannot manage the objects owned by the role***

![image info](https://docs.snowflake.com/en/_images/primary-secondary-roles-operations.png)

In [ ]:
%%sql -r dataframe_6
-- Grant privileges

GRANT USAGE ON DATABASE MARKETING TO ROLE MARKETING_ADMIN;
GRANT USAGE ON SCHEMA MARKETING.SALES TO ROLE MARKETING_ADMIN;
GRANT SELECT ON TABLE  MARKETING.SALES.CAMPAIGN TO ROLE MARKETING_ADMIN;
GRANT INSERT ON TABLE  MARKETING.SALES.CAMPAIGN TO ROLE MARKETING_ADMIN;


In [ ]:
%%sql -r dataframe_8
use role sysadmin;
-- Assign warehouse can be done with sysadmin, since sysadmin is the owner of the warehouse
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE MARKETING_ADMIN;

show grants on warehouse COMPUTE_WH ;

In [ ]:
%%sql -r dataframe_7
use role sysadmin;
-- Create tables
GRANT CREATE TABLE ON SCHEMA MARKETING.SALES TO ROLE MARKETING_ADMIN;

-- -- SELECT on all tables
GRANT SELECT ON ALL TABLES IN SCHEMA  MARKETING.SALES TO ROLE MARKETING_ADMIN;

-- -- SELECT on all future tables
GRANT SELECT ON FUTURE TABLES IN SCHEMA MARKETING.SALES TO ROLE MARKETING_ADMIN;


In [ ]:
%%sql -r dataframe_9
-- What are the privileges of the role?
SHOW GRANTS TO ROLE MARKETING_ADMIN;



In [ ]:
%%sql -r dataframe_10
-- To whom was the role assigned?
SHOW GRANTS OF ROLE MARKETING_ADMIN;


In [ ]:
%%sql -r dataframe_11
-- Drop USER and Database
USE ROLE SYSADMIN;
DROP DATABASE MARKETING;

In [ ]:
%%sql -r dataframe_12

USE ROLE SECURITYADMIN;
DROP ROLE MARKETING_ADMIN;
DROP USER INITIAL_USER;

## Check Grants

    show grants to user USER1;


In [ ]:
show users;

In [ ]:
%%sql -r dataframe_17
show grants to user MBENICIO19;

# Privileges 
| Object Type        | Main Privileges | Description |
|--------------------|-----------------|-------------|
| **ACCOUNT**        | CREATE DATABASE, CREATE USER, CREATE WAREHOUSE, CREATE INTEGRATION, MONITOR, IMPORTED PRIVILEGES | Controls global/account-level capabilities. |
| **DATABASE**       | CREATE SCHEMA, CREATE TABLE, USAGE, MODIFY | Allows creation of schemas and objects inside the DB. |
| **SCHEMA**         | CREATE TABLE, CREATE VIEW, CREATE STAGE, CREATE FILE FORMAT, USAGE | Controls object creation within schemas. |
| **TABLE**          | SELECT, INSERT, UPDATE, DELETE, TRUNCATE, REFERENCES, OWNERSHIP | Standard DML and reference constraints. |
| **VIEW**           | SELECT, REFERENCES, OWNERSHIP | Query access and dependency referencing. |
| **MATERIALIZED VIEW** | SELECT, REFRESH, REFERENCES, OWNERSHIP | Allows reading and refreshing MV data. |
| **DYNAMIC TABLE**  | SELECT, OPERATE, OWNERSHIP, INSERT/UPDATE (via target) | Dynamic-table–specific privileges (Snowflake documents separate privilege rules). |
| **STAGE**          | READ, WRITE, USAGE, OWNERSHIP | Controls file load/unload to/from stage. |
| **FILE FORMAT**    | USAGE, OWNERSHIP | Needed to use file formats in COPY INTO. |
| **SEQUENCE**       | USAGE, OWNERSHIP | Allows generating sequence values. |
| **FUNCTION** (UDF) | USAGE, OWNERSHIP | Allows calling UDFs. |
| **PROCEDURE**      | USAGE, OWNERSHIP | Allows executing stored procedures. |
| **PIPE**           | MONITOR, OPERATE, OWNERSHIP | Start, stop, or monitor Snowpipe. |
| **TASK**           | MONITOR, OPERATE, OWNERSHIP | Resume, suspend, and view task runs. |
| **WAREHOUSE**      | USAGE, OPERATE, MONITOR, MODIFY, OWNERSHIP | Controls query execution and warehouse management. |
| **STREAM**         | SELECT, USAGE, OWNERSHIP | Needed to read CDC data from streams. |
| **MASKING POLICY** | APPLY, OWNERSHIP | Allows applying column-level masking rules. |
| **ROW ACCESS POLICY** | APPLY, OWNERSHIP | Controls row-level filtering rules. |
| **TAG**            | APPLY, OWNERSHIP | Required to tag objects or enforce tag-based governance. |
| **RESOURCE MONITOR** | MONITOR, MODIFY, OWNERSHIP | Credit limit and warehouse monitoring. |
| **INTEGRATION** (e.g., STORAGE, API) | USAGE, OWNERSHIP | Required for external services (S3, GCS, Azure Blob, external functions). |